# Tech Challenge Fase 3 — Feature Engineering

**Objetivo:** Carregar as tabelas da camada Silver, filtradas por `ano = 2024`, e construir a base analítica para modelagem.

**Tabelas carregadas:**
| Tabela | Descrição |
|---|---|
| `workspace.silver.tc02_alunos` | Dados individuais de alunos |
| `workspace.silver.tc02_dim_municipio` | Dimensão município |
| `workspace.silver.tc02_dim_uf` | Dimensão UF |
| `workspace.silver.tc02_meta_brasil` | Metas nacionais de alfabetização |
| `workspace.silver.tc02_meta_municipio` | Metas por município |
| `workspace.silver.tc02_meta_uf` | Metas por UF |

## 0. Setup

In [1]:
import os
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Imports OK')

Imports OK


---
## 1. Conexão com o Databricks

In [2]:
import dotenv
from databricks import sql

dotenv.load_dotenv()

ACCESS_TOKEN    = os.getenv('SQL_DATABRICKS_ACCESS_TOKEN')
SERVER_HOSTNAME = os.getenv('SERVER_HOSTNAME')
WAREHOUSE_ID    = os.getenv('WAREHOUSE_ID')

def run_query(query: str) -> pd.DataFrame:
    with sql.connect(
        server_hostname=SERVER_HOSTNAME,
        http_path=f'/sql/1.0/warehouses/{WAREHOUSE_ID}',
        access_token=ACCESS_TOKEN,
    ) as conn:
        with conn.cursor() as cur:
            cur.execute(query)
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

print('Conexão configurada.')
print(f'  SERVER_HOSTNAME: {SERVER_HOSTNAME}')
print(f'  WAREHOUSE_ID:    {WAREHOUSE_ID}')
print(f'  ACCESS_TOKEN:    {"OK" if ACCESS_TOKEN else "NÃO ENCONTRADO"}')

Conexão configurada.
  SERVER_HOSTNAME: dbc-e8f1b9b5-37ed.cloud.databricks.com
  WAREHOUSE_ID:    9b3e392f288ad0f0
  ACCESS_TOKEN:    OK


---
## 2. Carregamento das Tabelas Silver (ano = 2024)

Cada tabela é lida com filtro `ano = 2024` diretamente na query SQL para minimizar o volume transferido.  
Resultados são salvos em parquet em `../data/silver/` para reuso sem nova conexão ao Databricks.

In [3]:
DATA_DIR = '../data/silver'
os.makedirs(DATA_DIR, exist_ok=True)

ANO_FILTRO = 2024

def load_table(table_name: str, extra_filter: str = '') -> pd.DataFrame:
    parquet_path = os.path.join(DATA_DIR, f'{table_name.split(".")[-1]}.parquet')
    if os.path.exists(parquet_path):
        df = pd.read_parquet(parquet_path)
        print(f'[cache] {table_name}: {df.shape}')
        return df

    where = f'WHERE ano = {ANO_FILTRO}'
    if extra_filter:
        where += f' AND {extra_filter}'

    query = f'SELECT * FROM {table_name} {where}'
    print(f'[databricks] Carregando: {table_name} ...')
    df = run_query(query)
    df.to_parquet(parquet_path, index=False)
    print(f'[databricks] {table_name}: {df.shape} → salvo em {parquet_path}')
    return df

print('Função load_table OK')

Função load_table OK


### 2.1 tc02_alunos

In [4]:
df_alunos = load_table('workspace.silver.tc02_alunos')

print(f'\nShape: {df_alunos.shape}')
print(f'Colunas: {df_alunos.columns.tolist()}')
display(df_alunos.head(3))

[databricks] Carregando: workspace.silver.tc02_alunos ...
[databricks] workspace.silver.tc02_alunos: (1851852, 16) → salvo em ../data/silver\tc02_alunos.parquet

Shape: (1851852, 16)
Colunas: ['ano', 'id_municipio', 'id_escola', 'id_aluno', 'caderno', 'serie', 'rede', 'presenca', 'preenchimento_caderno', 'alfabetizado', 'proficiencia', 'peso_prova_portugues', '_ano_ingestao', '_mes_ingestao', '_data_ingestao', '_data_processamento']


,ano,id_municipio,id_escola,id_aluno,caderno,serie,rede,presenca,preenchimento_caderno,alfabetizado,proficiencia,peso_prova_portugues,_ano_ingestao,_mes_ingestao,_data_ingestao,_data_processamento
0,2024,2104404,60005166,21046766,1,2,3,1,1,0,708.8400,1.0000,2026,9,2026-09-01 03:02:10.708704+00:00,2026-09-02 05:02:16.061001+00:00
1,2024,3131208,60021317,31141610,1,2,3,1,1,0,737.5000,1.0000,2026,9,2026-09-01 03:02:10.708704+00:00,2026-09-02 05:02:16.061001+00:00
2,2024,2303709,60009108,23057658,1,2,3,1,1,0,705.0100,1.0000,2026,9,2026-09-01 03:02:10.708704+00:00,2026-09-02 05:02:16.061001+00:00


### 2.2 tc02_dim_municipio

In [5]:
df_dim_municipio = load_table('workspace.silver.tc02_dim_municipio')

print(f'\nShape: {df_dim_municipio.shape}')
print(f'Colunas: {df_dim_municipio.columns.tolist()}')
display(df_dim_municipio.head(3))

[databricks] Carregando: workspace.silver.tc02_dim_municipio ...
[databricks] workspace.silver.tc02_dim_municipio: (12448, 19) → salvo em ../data/silver\tc02_dim_municipio.parquet

Shape: (12448, 19)
Colunas: ['ano', 'id_municipio', 'serie', 'rede', 'taxa_alfabetizacao', 'media_portugues', 'proporcao_aluno_nivel_0', 'proporcao_aluno_nivel_1', 'proporcao_aluno_nivel_2', 'proporcao_aluno_nivel_3', 'proporcao_aluno_nivel_4', 'proporcao_aluno_nivel_5', 'proporcao_aluno_nivel_6', 'proporcao_aluno_nivel_7', 'proporcao_aluno_nivel_8', '_ano_ingestao', '_mes_ingestao', '_data_ingestao', '_data_processamento']


,ano,id_municipio,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,_ano_ingestao,_mes_ingestao,_data_ingestao,_data_processamento
0,2024,1712702,2,3,9.0900,685.0300,27.2700,22.7300,9.0900,22.7300,13.6400,4.5500,0.0000,0.0000,0.0000,2026,9,2026-09-01 03:02:04.708764+00:00,2026-09-02 05:02:02.879854+00:00
1,2024,2919900,2,5,12.5000,695.9400,10.0000,20.0000,25.0000,30.0000,2.5000,7.5000,5.0000,0.0000,0.0000,2026,9,2026-09-01 03:02:04.708764+00:00,2026-09-02 05:02:02.879854+00:00
2,2024,1304260,2,3,11.2500,702.5900,7.8200,15.5900,29.6300,12.1800,25.6800,8.4200,0.6800,0.0000,0.0000,2026,9,2026-09-01 03:02:04.708764+00:00,2026-09-02 05:02:02.879854+00:00


### 2.3 tc02_dim_uf

In [6]:
df_dim_uf = load_table('workspace.silver.tc02_dim_uf')

print(f'\nShape: {df_dim_uf.shape}')
print(f'Colunas: {df_dim_uf.columns.tolist()}')
display(df_dim_uf.head(3))

[databricks] Carregando: workspace.silver.tc02_dim_uf ...
[databricks] workspace.silver.tc02_dim_uf: (75, 19) → salvo em ../data/silver\tc02_dim_uf.parquet

Shape: (75, 19)
Colunas: ['ano', 'sigla_uf', 'serie', 'rede', 'taxa_alfabetizacao', 'media_portugues', 'proporcao_aluno_nivel_0', 'proporcao_aluno_nivel_1', 'proporcao_aluno_nivel_2', 'proporcao_aluno_nivel_3', 'proporcao_aluno_nivel_4', 'proporcao_aluno_nivel_5', 'proporcao_aluno_nivel_6', 'proporcao_aluno_nivel_7', 'proporcao_aluno_nivel_8', '_ano_ingestao', '_mes_ingestao', '_data_ingestao', '_data_processamento']


,ano,sigla_uf,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,_ano_ingestao,_mes_ingestao,_data_ingestao,_data_processamento
0,2024,ES,2,3,70.9000,757.7300,1.5800,2.7400,5.9800,10.4300,13.7900,29.3600,24.1500,8.6200,3.3600,2026,9,2026-09-01 03:02:23.202544+00:00,2026-09-02 05:01:57.657994+00:00
1,2024,MT,2,3,61.0300,753.8000,1.8900,4.1700,8.1500,13.7700,15.8900,19.3700,20.7000,10.9800,5.0800,2026,9,2026-09-01 03:02:23.202544+00:00,2026-09-02 05:01:57.657994+00:00
2,2024,SP,2,5,58.1300,749.9000,3.0600,4.9300,8.5900,13.1500,17.3500,19.5700,16.7900,11.5400,5.0000,2026,9,2026-09-01 03:02:23.202544+00:00,2026-09-02 05:01:57.657994+00:00


### 2.4 tc02_meta_brasil

In [7]:
df_meta_brasil = load_table('workspace.silver.tc02_meta_brasil')

print(f'\nShape: {df_meta_brasil.shape}')
print(f'Colunas: {df_meta_brasil.columns.tolist()}')
display(df_meta_brasil.head(3))

[databricks] Carregando: workspace.silver.tc02_meta_brasil ...
[databricks] workspace.silver.tc02_meta_brasil: (1, 15) → salvo em ../data/silver\tc02_meta_brasil.parquet

Shape: (1, 15)
Colunas: ['ano', 'rede', 'taxa_alfabetizacao', 'meta_alfabetizacao_2024', 'meta_alfabetizacao_2025', 'meta_alfabetizacao_2026', 'meta_alfabetizacao_2027', 'meta_alfabetizacao_2028', 'meta_alfabetizacao_2029', 'meta_alfabetizacao_2030', 'percentual_participacao', '_ano_ingestao', '_mes_ingestao', '_data_ingestao', '_data_processamento']


,ano,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,percentual_participacao,_ano_ingestao,_mes_ingestao,_data_ingestao,_data_processamento
0,2024,PÚBLICA,59.2000,59.9000,63.7700,67.4700,70.9700,74.2300,77.2400,80.0000,87.3700,2026,9,2026-09-01 03:01:32.493675+00:00,2026-09-02 05:02:06.224619+00:00


### 2.5 tc02_meta_municipio

In [8]:
df_meta_municipio = load_table('workspace.silver.tc02_meta_municipio')

print(f'\nShape: {df_meta_municipio.shape}')
print(f'Colunas: {df_meta_municipio.columns.tolist()}')
display(df_meta_municipio.head(3))

[databricks] Carregando: workspace.silver.tc02_meta_municipio ...
[databricks] workspace.silver.tc02_meta_municipio: (5352, 17) → salvo em ../data/silver\tc02_meta_municipio.parquet

Shape: (5352, 17)
Colunas: ['ano', 'id_municipio', 'rede', 'taxa_alfabetizacao', 'meta_alfabetizacao_2024', 'meta_alfabetizacao_2025', 'meta_alfabetizacao_2026', 'meta_alfabetizacao_2027', 'meta_alfabetizacao_2028', 'meta_alfabetizacao_2029', 'meta_alfabetizacao_2030', 'nivel_alfabetizacao', 'percentual_participacao', '_ano_ingestao', '_mes_ingestao', '_data_ingestao', '_data_processamento']


,ano,id_municipio,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,nivel_alfabetizacao,percentual_participacao,_ano_ingestao,_mes_ingestao,_data_ingestao,_data_processamento
0,2024,3167509,MUNICIPAL,48.2500,14.1400,21.8900,32.2900,44.8000,58.0000,70.1500,80.0000,1,88.8900,2026,9,2026-09-01 03:01:59.184593+00:00,2026-09-02 05:02:12.527807+00:00
1,2024,2400604,MUNICIPAL,21.4300,17.0200,25.1700,35.5700,47.5300,59.7700,70.9100,80.0000,0,100.0000,2026,9,2026-09-01 03:01:59.184593+00:00,2026-09-02 05:02:12.527807+00:00
2,2024,5004809,MUNICIPAL,41.7600,18.9100,27.2500,37.5600,49.1300,60.8000,71.3500,80.0000,1,86.9200,2026,9,2026-09-01 03:01:59.184593+00:00,2026-09-02 05:02:12.527807+00:00


### 2.6 tc02_meta_uf

In [9]:
df_meta_uf = load_table('workspace.silver.tc02_meta_uf')

print(f'\nShape: {df_meta_uf.shape}')
print(f'Colunas: {df_meta_uf.columns.tolist()}')
display(df_meta_uf.head(3))

[databricks] Carregando: workspace.silver.tc02_meta_uf ...
[databricks] workspace.silver.tc02_meta_uf: (26, 16) → salvo em ../data/silver\tc02_meta_uf.parquet

Shape: (26, 16)
Colunas: ['ano', 'sigla_uf', 'rede', 'taxa_alfabetizacao', 'meta_alfabetizacao_2024', 'meta_alfabetizacao_2025', 'meta_alfabetizacao_2026', 'meta_alfabetizacao_2027', 'meta_alfabetizacao_2028', 'meta_alfabetizacao_2029', 'meta_alfabetizacao_2030', 'percentual_participacao', '_ano_ingestao', '_mes_ingestao', '_data_ingestao', '_data_processamento']


,ano,sigla_uf,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,percentual_participacao,_ano_ingestao,_mes_ingestao,_data_ingestao,_data_processamento
0,2024,AL,PÚBLICA,48.6300,49.7000,55.5000,61.1000,66.5000,71.5000,76.0000,80.0000,93.7800,2026,9,2026-09-01 03:01:53.345372+00:00,2026-09-02 05:02:09.685465+00:00
1,2024,MT,PÚBLICA,60.5900,59.2000,63.2000,67.0000,70.7000,74.0000,77.2000,80.0000,88.4600,2026,9,2026-09-01 03:01:53.345372+00:00,2026-09-02 05:02:09.685465+00:00
2,2024,PR,PÚBLICA,70.4200,74.2000,75.2000,76.2000,77.2000,78.2000,79.1000,80.0000,86.2500,2026,9,2026-09-01 03:01:53.345372+00:00,2026-09-02 05:02:09.685465+00:00


---
## 3. Visão Geral das Tabelas Carregadas

In [10]:
tables = {
    'tc02_alunos':         df_alunos,
    'tc02_dim_municipio':  df_dim_municipio,
    'tc02_dim_uf':         df_dim_uf,
    'tc02_meta_brasil':    df_meta_brasil,
    'tc02_meta_municipio': df_meta_municipio,
    'tc02_meta_uf':        df_meta_uf,
}

summary_rows = []
for name, df in tables.items():
    summary_rows.append({
        'tabela':        name,
        'linhas':        df.shape[0],
        'colunas':       df.shape[1],
        'nulos_total':   df.isnull().sum().sum(),
        'colunas_lista': ', '.join(df.columns.tolist()),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df[['tabela', 'linhas', 'colunas', 'nulos_total']])

,tabela,linhas,colunas,nulos_total
0,tc02_alunos,1851852,16,0
1,tc02_dim_municipio,12448,19,0
2,tc02_dim_uf,75,19,0
3,tc02_meta_brasil,1,15,0
4,tc02_meta_municipio,5352,17,120
5,tc02_meta_uf,26,16,2


In [11]:
df_alunos

,ano,id_municipio,id_escola,id_aluno,caderno,serie,rede,presenca,preenchimento_caderno,alfabetizado,proficiencia,peso_prova_portugues,_ano_ingestao,_mes_ingestao,_data_ingestao,_data_processamento
0,2024,2104404,60005166,21046766,1,2,3,1,1,0,708.8400,1.0000,2026,9,2026-09-01 03:02:10.708704+00:00,2026-09-02 05:02:16.061001+00:00
1,2024,3131208,60021317,31141610,1,2,3,1,1,0,737.5000,1.0000,2026,9,2026-09-01 03:02:10.708704+00:00,2026-09-02 05:02:16.061001+00:00
2,2024,2303709,60009108,23057658,1,2,3,1,1,0,705.0100,1.0000,2026,9,2026-09-01 03:02:10.708704+00:00,2026-09-02 05:02:16.061001+00:00
3,2024,2613008,60013145,26013332,1,2,3,1,1,1,775.9800,1.0000,2026,9,2026-09-01 03:02:10.708704+00:00,2026-09-02 05:02:16.061001+00:00
4,2024,2914604,60015020,29101498,1,2,3,1,1,0,668.6100,1.0000,2026,9,2026-09-01 03:02:10.708704+00:00,2026-09-02 05:02:16.061001+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1851847,2024,1200401,60000496,12008208,9,2,2,1,1,1,750.8200,1.5400,2026,9,2026-09-01 03:02:10.708704+00:00,2026-09-02 05:02:16.061001+00:00
1851848,2024,3300209,60025577,33003824,9,2,3,1,1,0,726.0700,1.5600,2026,9,2026-09-01 03:02:10.708704+00:00,2026-09-02 05:02:16.061001+00:00
1851849,2024,4316402,60038258,43073020,9,2,2,1,1,0,693.9600,1.5714,2026,9,2026-09-01 03:02:10.708704+00:00,2026-09-02 05:02:16.061001+00:00
1851850,2024,4314902,60038008,43062505,9,2,2,1,1,0,731.7100,1.6667,2026,9,2026-09-01 03:02:10.708704+00:00,2026-09-02 05:02:16.061001+00:00


In [76]:
columns_to_select_mun = ['id_municipio', 'serie', 'rede', 'sigla_uf', 'taxa_alfabetizacao', 'media_portugues']

In [22]:
# Mapa UF: código IBGE (2 dígitos) → sigla e nome completo
_UF_SIGLA: dict = {
    "11": "RO", "12": "AC", "13": "AM", "14": "RR", "15": "PA",
    "16": "AP", "17": "TO", "21": "MA", "22": "PI", "23": "CE",
    "24": "RN", "25": "PB", "26": "PE", "27": "AL", "28": "SE",
    "29": "BA", "31": "MG", "32": "ES", "33": "RJ", "35": "SP",
    "41": "PR", "42": "SC", "43": "RS", "50": "MS", "51": "MT",
    "52": "GO", "53": "DF",
}

In [77]:
# Chave: <id_municipio, serie, rede>
# df_dim_municipio['chave'] = df_dim_municipio.apply(lambda row: f"{row['id_municipio']}_{row['serie']}_{row['rede']}", axis=1)
df_dim_municipio['cod_uf'] = df_dim_municipio['id_municipio'].astype(str).str[:2]
df_dim_municipio['sigla_uf'] = df_dim_municipio['cod_uf'].map(_UF_SIGLA)
df_dim_municipio_final = df_dim_municipio[columns_to_select_mun]
df_dim_municipio_final.columns = ['id_municipio', 'serie', 'rede', 'sigla_uf', 'taxa_alfabetizacao_mun', 'media_portugues_mun']

In [79]:
columns_to_select_uf = ['sigla_uf', 'serie', 'rede', 'taxa_alfabetizacao', 'media_portugues']

In [80]:
# Chave: <sigla_uf, serie, rede>
# df_dim_uf['chave'] = df_dim_uf.apply(lambda row: f"{row['sigla_uf']}_{row['serie']}_{row['rede']}", axis=1)
df_dim_uf_final = df_dim_uf[columns_to_select_uf]
df_dim_uf_final.columns = ['sigla_uf', 'serie', 'rede', 'taxa_alfabetizacao_uf', 'media_portugues_uf']

In [81]:
df_dim_uf.head()

,ano,sigla_uf,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,_ano_ingestao,_mes_ingestao,_data_ingestao,_data_processamento,chave
0,2024,ES,2,3,70.9000,757.7300,1.5800,2.7400,5.9800,10.4300,13.7900,29.3600,24.1500,8.6200,3.3600,2026,9,2026-09-01 03:02:23.202544+00:00,2026-09-02 05:01:57.657994+00:00,ES_2_3
1,2024,MT,2,3,61.0300,753.8000,1.8900,4.1700,8.1500,13.7700,15.8900,19.3700,20.7000,10.9800,5.0800,2026,9,2026-09-01 03:02:23.202544+00:00,2026-09-02 05:01:57.657994+00:00,MT_2_3
2,2024,SP,2,5,58.1300,749.9000,3.0600,4.9300,8.5900,13.1500,17.3500,19.5700,16.7900,11.5400,5.0000,2026,9,2026-09-01 03:02:23.202544+00:00,2026-09-02 05:01:57.657994+00:00,SP_2_5
3,2024,PB,2,2,54.9700,747.3900,3.3200,5.5000,9.2500,15.4500,16.6400,17.0400,16.3600,12.3000,4.1300,2026,9,2026-09-01 03:02:23.202544+00:00,2026-09-02 05:01:57.657994+00:00,PB_2_2
4,2024,AP,2,5,46.6200,739.8300,3.3700,6.7600,11.5000,17.7300,18.8700,15.9600,13.6500,8.1900,3.9800,2026,9,2026-09-01 03:02:23.202544+00:00,2026-09-02 05:01:57.657994+00:00,AP_2_5


In [45]:
df_meta_municipio.groupby('nivel_alfabetizacao').agg(
    taxa_media=('taxa_alfabetizacao', 'mean'),
    taxa_min=('taxa_alfabetizacao', 'min'),
    taxa_max=('taxa_alfabetizacao', 'max')
)

,taxa_media,taxa_min,taxa_max
nivel_alfabetizacao,,,
0,31.6122,4.4000,39.9900
1,45.2129,40.0000,49.9600
2,55.1078,50.0000,59.9700
3,65.0508,60.0000,69.9800
4,74.9111,70.0000,79.9700
5,88.4026,80.0000,100.0000


In [46]:
df_meta_municipio.groupby('nivel_alfabetizacao').agg(
    pct_media=('percentual_participacao', 'mean'),
    pct_min=('percentual_participacao', 'min'),
    pct_max=('percentual_participacao', 'max')
)

,pct_media,pct_min,pct_max
nivel_alfabetizacao,,,
0,88.9376,70.0000,100.0000
1,89.0415,70.3200,100.0000
2,90.1587,70.0400,100.0000
3,90.7691,70.2500,100.0000
4,91.3103,70.0000,100.0000
5,94.2123,70.9700,100.0000


In [82]:
df_consolidado = pd.merge(df_alunos, df_dim_municipio_final, on=['id_municipio', 'serie', 'rede'], how='left').merge(df_dim_uf_final, on=['sigla_uf', 'serie', 'rede'], how='left')

In [139]:
int_cols = ['id_municipio', 'id_escola', 'id_aluno', 'serie', 'rede', 'alfabetizado', 'caderno']

for col in int_cols:
    df_consolidado[col] = pd.to_numeric(df_consolidado[col], errors='raise').astype('int64')

df_consolidado[int_cols].dtypes

id_municipio    int64
id_escola       int64
id_aluno        int64
serie           int64
rede            int64
alfabetizado    int64
caderno         int64
dtype: object

In [151]:
df_consolidado = df_consolidado[df_consolidado['caderno'] < 20]

In [152]:
columns_to_drop = ['presenca', 'preenchimento_caderno', 'caderno']

In [153]:
df_consolidado_final = df_consolidado.drop(columns=columns_to_drop)

In [159]:
DATA_DIR = '../data/'

parquet_path = os.path.join(DATA_DIR, 'df_consolidado.parquet')

df_consolidado_final.to_parquet(parquet_path, index=False)